# Libraries

In [162]:
import pandas as pd
import seaborn as sns
from functools import reduce

# Resampling Libs
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

# Functions

## Fun_Resample

In [163]:
def fun_res(dfi):

    X = dfi.drop("Study_Status_Bin", axis = 1)
    y = dfi["Study_Status_Bin"]

    X_train_tts, X_test_tts, y_train_tts, y_test_tts = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

    res = RandomUnderSampler(sampling_strategy = 'auto', random_state = 42)
    X_train_res, y_train_res = res.fit_resample(X_train_tts, y_train_tts) 
    
    X_train_res = pd.DataFrame(X_train_res)

    return X_train_res, y_train_res


## Fun_Sparse

In [164]:
def fun_sparse(i, dfi, categ_cols):
    pivot_tables = []

    for col in categ_cols:
        if col not in dfi.columns:
            # αν δεν υπάρχει, φτιάξε dummy στήλη γεμάτη NaN/0
            dfi[col] = pd.Series([0]*len(dfi), index=dfi.index)

        pivot_table = pd.pivot_table(
            data = dfi,
            index = col,
            columns = "Study_Status_Bin",
            aggfunc = "size",
            fill_value = 0,
            observed = False
        ).reset_index()

        # Change Column/Element Names of Pivot
        pivot_table['Variables'] = col + ' = ' + pivot_table[col].astype(str)
        pivot_table.drop(columns=[col], inplace=True)

        # Reindex Column of value
        final_cols = ['Variables'] + [c for c in pivot_table.columns if c != 'Variables']
        pivot_table = pivot_table[final_cols]

        pivot_tables.append(pivot_table)

    # Merge all pivots
    pivot_merged = pd.concat(pivot_tables, ignore_index=True)
    pivot_merged = pivot_merged.rename(columns={0: f'0 - df{i}', 1: f'1 - df{i}'})
    # pivot_merged = pivot_merged.drop(columns = [f'0 - df{i}'], axis = 1) --> EPV regarding the minority (1) class only
    # However, the variables are the same even if include majority (0) in pivots --> keep majority for exploratory analysis
    
    return pivot_merged


## Fun_Zeros

In [165]:
def fun_zeros(pivot_merged, count, missing):

    num_cols = pivot_merged.select_dtypes(include='number')
    mask = num_cols < count
    if missing == True:
        mask |= num_cols.isna()
    sparse = pivot_merged[mask.any(axis=1)]
    return sparse

# Load Unmerged Data
- Data loaded haven't dropped first after dummies. All levels need checking
- Only on train data. 
- Test data are 'unseen' --> no sparsity check. 

In [166]:
df1 = pd.read_pickle(r".\df_dummies_unmerged\df1_dummies_unmerged.pkl")
df2 = pd.read_pickle(r".\df_dummies_unmerged\df2_dummies_unmerged.pkl")
df3 = pd.read_pickle(r".\df_dummies_unmerged\df3_dummies_unmerged.pkl")
df4 = pd.read_pickle(r".\df_dummies_unmerged\df4_dummies_unmerged.pkl")

In [167]:
# Will be dropped as they will not be used and not needed for check
df1 = df1.drop([col for col in df1 if 'Adverse_System' in col or 'Masking_Detail' in col or 'Covid' in col], axis = 1)
df2 = df2.drop([col for col in df2 if 'Adverse_System' in col or 'Masking_Detail' in col or 'Covid' in col], axis = 1)
df3 = df3.drop([col for col in df3 if 'Adverse_System' in col or 'Masking_Detail' in col or 'Covid' in col], axis = 1)
df4 = df4.drop([col for col in df4 if 'Adverse_System' in col or 'Masking_Detail' in col or 'Covid' in col], axis = 1)

## Resample

In [168]:
X_train1, y_train1 = fun_res(df1)
X_train2, y_train2 = fun_res(df2)
X_train3, y_train3 = fun_res(df3)
X_train4, y_train4 = fun_res(df4)
X_train4.head()

,Enrollment_Counts,Funder_Counts,Funder_Counts_Log,Intervention_Type_Counts,Intervention_Route_Counts,Placebo_Bin,Standard_Care_Bin,Healthy_Bin,Adverse_Counts,Adverse_Counts_Log,...,Primary_Purpose_List_SCREENING,Primary_Purpose_List_SUPPORTIVE_CARE,Primary_Purpose_List_TREATMENT,Continents_List_Africa,Continents_List_Asia,Continents_List_Cont_Other,Continents_List_Europe,Continents_List_North America,Continents_List_Oceania,Continents_List_South America
1933,454,1,0.693147,1,1,0,0,0,27,3.332205,...,0,0,1,0,0,0,1,0,0,0
7496,64,3,1.386294,1,1,1,0,1,0,0.000000,...,0,1,0,0,0,1,0,0,0,0
3966,619,1,0.693147,1,1,1,0,0,0,0.000000,...,0,0,0,0,1,0,0,0,0,0
2389,211,2,1.098612,1,1,1,0,0,0,0.000000,...,0,0,0,0,0,0,1,0,0,0
8416,612,2,1.098612,1,1,0,0,1,0,0.000000,...,0,0,0,0,1,0,0,0,0,0


## Create X_y dfs

In [169]:
df1_train = pd.concat([X_train1, y_train1], axis=1)  # X_train y_train have same index
df2_train = pd.concat([X_train2, y_train2], axis=1)  # X_train y_train have same index
df3_train = pd.concat([X_train3, y_train3], axis=1)  # X_train y_train have same index
df4_train = pd.concat([X_train4, y_train4], axis=1)  # X_train y_train have same index

display(X_train1.shape)
display(X_train2.shape)
display(X_train3.shape)
display(X_train4.shape)

(5092, 137)

(8678, 138)

(3854, 136)

(3694, 137)

## Unique cols

In [170]:
dfs = [X_train1, X_train2, X_train3, X_train4]
all_unique_cols = set().union(*(df.columns for df in dfs))
print(f"{len(all_unique_cols)}")

# Columns missing from the dfs
for i, df in enumerate(dfs, start=1):
    missing = all_unique_cols - set(df.columns)
    extra = set(df.columns) - all_unique_cols
    print(f" X_train{i}: shape={df.shape}")
    print(f" Missing cols: {missing if missing else 'None'}")
    print(f" Extra cols:   {extra if extra else 'None'}\n")


140
 X_train1: shape=(5092, 137)
 Missing cols: {'Conditions_Detail_List_Chemical Actions and Uses', 'Conditions_Detail_List_Fluids and Secretions', 'Conditions_Detail_List_Hemic and Immune Systems'}
 Extra cols:   None

 X_train2: shape=(8678, 138)
 Missing cols: {'Conditions_Detail_List_Chemical Actions and Uses', 'Conditions_Detail_List_Fluids and Secretions'}
 Extra cols:   None

 X_train3: shape=(3854, 136)
 Missing cols: {'Conditions_Detail_List_Chemical Actions and Uses', 'Conditions_Detail_List_Information Science', 'Conditions_Detail_List_Hemic and Immune Systems', 'Conditions_Detail_List_Human Activities'}
 Extra cols:   None

 X_train4: shape=(3694, 137)
 Missing cols: {'Conditions_Detail_List_Human Activities', 'Conditions_Detail_List_Fluids and Secretions', 'Conditions_Detail_List_Hemic and Immune Systems'}
 Extra cols:   None



## Pivots

### List/Categ/Bin Pivot

In [171]:
# Categ Pivot
categ_cols = [col for col in all_unique_cols if '_Categ' in col or '_Bin' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, categ_cols)
pivot_merged2 = fun_sparse(2, df2_train, categ_cols)
pivot_merged3 = fun_sparse(3, df3_train, categ_cols)
pivot_merged4 = fun_sparse(4, df4_train, categ_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_categ = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_categ

# List Pivot
list_cols = [col for col in all_unique_cols if '_List' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, list_cols)
pivot_merged2 = fun_sparse(2, df2_train, list_cols)
pivot_merged3 = fun_sparse(3, df3_train, list_cols)
pivot_merged4 = fun_sparse(4, df4_train, list_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_list = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_list


#### Zeros Categ/Bin


In [172]:
sparse_categ_train = fun_zeros(pivot_merged_train_categ, 25, False) # No categorical cols were used. 
display(sparse_categ_train)

sparse_list_train = fun_zeros(pivot_merged_train_list, 25, True)
display(sparse_list_train)

Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4


Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4
19,Conditions_Detail_List_Biological Phenomena = 1,2.0,1.0,3.0,5.0,0.0,2.0,3.0,4.0
23,Conditions_Detail_List_Cell Physiological Phen...,3.0,4.0,13.0,17.0,8.0,13.0,19.0,9.0
25,Conditions_Detail_List_Cells = 1,1.0,1.0,2.0,4.0,NaN,NaN,NaN,NaN
27,Conditions_Detail_List_Chemical Actions and Us...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
31,Conditions_Detail_List_Circulatory and Respira...,NaN,NaN,3.0,4.0,2.0,0.0,0.0,6.0
37,Conditions_Detail_List_Diagnosis = 1,30.0,13.0,61.0,39.0,25.0,15.0,39.0,36.0
41,Conditions_Detail_List_Education = 1,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN
45,Conditions_Detail_List_Environment and Public ...,1.0,2.0,10.0,11.0,4.0,4.0,10.0,14.0
52,Conditions_Detail_List_Genetic Phenomena = 1,3.0,0.0,2.0,2.0,NaN,NaN,NaN,NaN
54,Conditions_Detail_List_Health Care Economics a...,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN


# Check on Merged

## Load Merged Data

In [173]:
df1 = pd.read_pickle(r".\df_dummies\df1_dummies.pkl")
df2 = pd.read_pickle(r".\df_dummies\df2_dummies.pkl")
df3 = pd.read_pickle(r".\df_dummies\df3_dummies.pkl")
df4 = pd.read_pickle(r".\df_dummies\df4_dummies.pkl")

In [174]:
df4.shape

(11704, 73)

## Functions

In [175]:
# Resample
X_train1, y_train1 = fun_res(df1)
X_train2, y_train2 = fun_res(df2)
X_train3, y_train3 = fun_res(df3)
X_train4, y_train4 = fun_res(df4)

# Create X_train y_train
df1_train = pd.concat([X_train1, y_train1], axis=1)  # X_train y_train have same index
df2_train = pd.concat([X_train2, y_train2], axis=1)  # X_train y_train have same index
df3_train = pd.concat([X_train3, y_train3], axis=1)  # X_train y_train have same index
df4_train = pd.concat([X_train4, y_train4], axis=1)  # X_train y_train have same index

# Unique Columns
dfs = [X_train1, X_train2, X_train3, X_train4]
all_unique_cols = set().union(*(df.columns for df in dfs))
print(f"{len(all_unique_cols)}")

for i, df in enumerate(dfs, start=1):
    missing = all_unique_cols - set(df.columns)
    extra = set(df.columns) - all_unique_cols
    # print(f" X_train{i}: shape={df.shape}")
    # print(f" Missing cols: {missing if missing else 'None'}")
    # print(f" Extra cols:   {extra if extra else 'None'}\n")

72


## Interaction

In [176]:
# Columns
inter_cols = [col for col in all_unique_cols if '_x_' in col and 'Enrollment' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, inter_cols)
pivot_merged2 = fun_sparse(2, df2_train, inter_cols)
pivot_merged3 = fun_sparse(3, df3_train, inter_cols)
pivot_merged4 = fun_sparse(4, df4_train, inter_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_inter = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_inter


### Zeros Inter


In [177]:
sparse_inter_train = fun_zeros(pivot_merged_train_inter, 25, True)
display(sparse_inter_train)

Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4


## Categ/Bin/List Pivot

In [178]:
# Pivot_Categ/Bin
categ_cols = [col for col in all_unique_cols if '_Categ' in col or '_Bin' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, categ_cols)
pivot_merged2 = fun_sparse(2, df2_train, categ_cols)
pivot_merged3 = fun_sparse(3, df3_train, categ_cols)
pivot_merged4 = fun_sparse(4, df4_train, categ_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_categ = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 

# Pivot_List
list_cols = [col for col in all_unique_cols if '_List' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, list_cols)
pivot_merged2 = fun_sparse(2, df2_train, list_cols)
pivot_merged3 = fun_sparse(3, df3_train, list_cols)
pivot_merged4 = fun_sparse(4, df4_train, list_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_list = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 


### Zeros Categ/Bin/List


In [181]:
sparse_categ_train = fun_zeros(pivot_merged_train_categ, 25, False) # No categorical cols were used. 
display(sparse_categ_train) # ok

sparse_list_train = fun_zeros(pivot_merged_train_list, 25, True)
display(sparse_list_train) # ok


Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4


Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4
19,Conditions_Detail_List_Eye = 1,24,22,76,62,41,39,29,31
41,"Conditions_Detail_List_Stomatognathic, Otorhin...",22,18,80,58,38,36,38,20
47,"Intervention_Model_List_FACTORIAL, SEQUENTIAL,...",219,387,73,107,30,21,33,19
77,Primary_Purpose_List_DIAGNOSTIC = 1,30,63,54,64,22,21,30,35
